<a href="https://colab.research.google.com/github/thunerous/MH/blob/main/Copy_of_LR_10_SIDORTSEV.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

pip install numpy==1.26.4
pip install scikit-surprise==1.1.4

Found existing installation: numpy 2.0.2
Uninstalling numpy-2.0.2:
  Successfully uninstalled numpy-2.0.2
Note: you may need to restart the kernel to use updated packages.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 227.5 MB/s eta 0:00:00a 0:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
kaggle-environments 1.27.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.


In [ ]:
import os
os._exit(0)

In [ ]:
import numpy as np
print(np.__version__)

1.26.4


In [ ]:
from surprise import Dataset, Reader
from surprise import SVD, KNNBasic
from surprise.model_selection import cross_validate, GridSearchCV, train_test_split
from surprise import accuracy

import numpy as np
print("NumPy:", np.__version__)
print("Surprise imported OK")

NumPy: 1.26.4
Surprise imported OK


!wget --no-check-certificate https://files.grouplens.org/datasets/movielens/ml-100k.zip
!unzip -o ml-100k.zip

import pandas as pd
import sys
import io
from surprise import Dataset, Reader, SVD, KNNBasic
from surprise.model_selection import cross_validate, GridSearchCV, train_test_split
from surprise import accuracy

data = pd.read_csv('ml-100k/u.data', sep='\t', names=['user_id', 'movie_id', 'rating', 'timestamp'])
print("Размер данных:", data.shape)
data.head()

reader = Reader(rating_scale=(1, 5))
data_surprise = Dataset.load_from_df(data[['user_id', 'movie_id', 'rating']], reader)
trainset, testset = train_test_split(data_surprise, test_size=0.2)

print(f"Train size: {trainset.n_users} users, {trainset.n_items} items")
print(f"Test size: {len(testset)} ratings")

In [ ]:
param_grid_svd = {
    'n_factors': [50, 100],
    'lr_all': [0.005, 0.010],
    'reg_all': [0.02, 0.05]
}

grid_search_svd = GridSearchCV(SVD, param_grid_svd, measures=['rmse', 'mae'], cv=3, n_jobs=-1)
grid_search_svd.fit(data_surprise)

print("Best RMSE for SVD:", grid_search_svd.best_score['rmse'])
print("Best params for SVD:", grid_search_svd.best_params['rmse'])
print("Best MAE for SVD:", grid_search_svd.best_score['mae'])

Best RMSE for SVD: 0.9297779271142651
Best params for SVD: {'n_factors': 50, 'lr_all': 0.01, 'reg_all': 0.05}
Best MAE for SVD: 0.7328367499766117


In [ ]:
param_grid_knn = {
    'k': [20, 40],
    'min_k': [1, 5],
    'sim_options': {
        'name': ['cosine', 'pearson'],
        'user_based': [True, False]
    }
}

sys_stdout_backup = sys.stdout
sys.stdout = io.StringIO()

grid_search_knn = GridSearchCV(KNNBasic, param_grid_knn, measures=['rmse', 'mae'], cv=3, n_jobs=-1)
grid_search_knn.fit(data_surprise)

sys.stdout = sys_stdout_backup

print("Best RMSE for KNNBasic:", grid_search_knn.best_score['rmse'])
print("Best params for KNNBasic:", grid_search_knn.best_params['rmse'])
print("Best MAE for KNNBasic:", grid_search_knn.best_score['mae'])

Best RMSE for KNNBasic: 1.019641170397384
Best params for KNNBasic: {'k': 40, 'min_k': 5, 'sim_options': {'name': 'pearson', 'user_based': True}}
Best MAE for KNNBasic: 0.8085823150289672


In [ ]:
best_svd_rmse = grid_search_svd.best_estimator['rmse']
best_knn_rmse = grid_search_knn.best_estimator['rmse']

best_svd_rmse.fit(trainset)
predictions_svd = best_svd_rmse.test(testset)
print("Test RMSE (SVD):", accuracy.rmse(predictions_svd))
print("Test MAE (SVD):", accuracy.mae(predictions_svd))

best_knn_rmse.fit(trainset)
predictions_knn = best_knn_rmse.test(testset)
print("\nTest RMSE (KNNBasic):", accuracy.rmse(predictions_knn))
print("Test MAE (KNNBasic):", accuracy.mae(predictions_knn))

RMSE: 0.9263
Test RMSE (SVD): 0.9262512148080372
MAE:  0.7316
Test MAE (SVD): 0.7315776538236488
Computing the pearson similarity matrix...
Done computing similarity matrix.
RMSE: 1.0214

Test RMSE (KNNBasic): 1.0214290973858615
MAE:  0.8126
Test MAE (KNNBasic): 0.812587385605733


In [ ]:
svd_mae_score = grid_search_svd.best_score['mae']
knn_mae_score = grid_search_knn.best_score['mae']

print(f"CV MAE для SVD: {svd_mae_score:.4f}")
print(f"CV MAE для KNNBasic: {knn_mae_score:.4f}")

if svd_mae_score < knn_mae_score:
    best_model = best_svd_rmse
    model_name = "SVD"
else:
    best_model = best_knn_rmse
    model_name = "KNNBasic"

print(f"\nВыбранный алгоритм: {model_name}")

CV MAE для SVD: 0.7328
CV MAE для KNNBasic: 0.8086

Выбранный алгоритм: SVD


In [ ]:
print("Примеры оценок в trainset:")
for i, (uid, iid, rating) in enumerate(trainset.all_ratings()):
    raw_uid = trainset.to_raw_uid(uid)
    raw_iid = trainset.to_raw_iid(iid)
    print(f"User {raw_uid}, Movie {raw_iid}, Rating: {rating}")
    if i >= 10:
        break

Примеры оценок в trainset:
User 70, Movie 501, Rating: 4.0
User 70, Movie 210, Rating: 4.0
User 70, Movie 151, Rating: 3.0
User 70, Movie 227, Rating: 3.0
User 70, Movie 228, Rating: 5.0
User 70, Movie 449, Rating: 2.0
User 70, Movie 298, Rating: 5.0
User 70, Movie 655, Rating: 4.0
User 70, Movie 197, Rating: 4.0
User 70, Movie 83, Rating: 4.0
User 70, Movie 383, Rating: 2.0


In [ ]:
best_model.fit(trainset)
user_id = data['user_id'].iloc[0]  # первый пользователь из датасета

try:
    inner_uid = trainset.to_inner_uid(user_id)
    user_ratings = trainset.ur[inner_uid]
    print(f"Количество рецензий пользователя {user_id}: {len(user_ratings)}")

    all_items = set(trainset.all_items())
    rated_items = set([item for (item, _) in user_ratings])
    unrated_items = all_items - rated_items

    predictions = [
        (item, best_model.predict(user_id, trainset.to_raw_iid(item)).est)
        for item in unrated_items
    ]

    predictions.sort(key=lambda x: x[1], reverse=True)

    print("Топ-10 фильмов, рекомендованных для пользователя:")
    for item_id, rating in predictions[:10]:
        print(f"Фильм {trainset.to_raw_iid(item_id)} с прогнозируемым рейтингом {rating:.2f}")

except ValueError:
    print(f"Пользователь {user_id} не найден в trainset")

Количество рецензий пользователя 196: 26
Топ-10 фильмов, рекомендованных для пользователя:
Фильм 318 с прогнозируемым рейтингом 4.64
Фильм 64 с прогнозируемым рейтингом 4.57
Фильм 178 с прогнозируемым рейтингом 4.51
Фильм 408 с прогнозируемым рейтингом 4.46
Фильм 1449 с прогнозируемым рейтингом 4.37
Фильм 657 с прогнозируемым рейтингом 4.36
Фильм 191 с прогнозируемым рейтингом 4.35
Фильм 603 с прогнозируемым рейтингом 4.35
Фильм 313 с прогнозируемым рейтингом 4.32
Фильм 1169 с прогнозируемым рейтингом 4.32


In [ ]:
user_id_to_check = 214
if user_id_to_check in data['user_id'].values:
    print(f"Пользователь {user_id_to_check} найден в данных")

    try:
        inner_uid = trainset.to_inner_uid(user_id_to_check)
        user_ratings = trainset.ur[inner_uid]
        print(f"Количество рецензий пользователя {user_id_to_check}: {len(user_ratings)}")

        all_items = set(trainset.all_items())
        rated_items = set([item for (item, _) in user_ratings])
        unrated_items = all_items - rated_items
        predictions = [(item, best_model.predict(user_id_to_check, trainset.to_raw_iid(item)).est)
                      for item in unrated_items]
        predictions.sort(key=lambda x: x[1], reverse=True)

        print("Топ-10 фильмов, рекомендованных для пользователя:")
        for item_id, rating in predictions[:10]:
            print(f"Фильм {trainset.to_raw_iid(item_id)} с прогнозируемым рейтингом {rating:.2f}")

    except ValueError:
        print(f"Пользователь {user_id_to_check} НЕ попал в trainset (возможно, из-за train_test_split)")
        print("Попробуем другого пользователя, который точно есть в trainset:")

        available_users = [trainset.to_raw_uid(uid) for uid in trainset.all_users()]
        print(f"Доступные пользователи в trainset (первые 10): {available_users[:10]}")

        alt_user_id = available_users[0]
        inner_uid = trainset.to_inner_uid(alt_user_id)
        user_ratings = trainset.ur[inner_uid]
        print(f"\nИспользуем пользователя {alt_user_id}")
        print(f"Количество рецензий: {len(user_ratings)}")

        all_items = set(trainset.all_items())
        rated_items = set([item for (item, _) in user_ratings])
        unrated_items = all_items - rated_items

        predictions = [(item, best_model.predict(alt_user_id, trainset.to_raw_iid(item)).est)
                      for item in unrated_items]
        predictions.sort(key=lambda x: x[1], reverse=True)

        print("\nТоп-10 фильмов, рекомендованных для пользователя:")
        for item_id, rating in predictions[:10]:
            print(f"Фильм {trainset.to_raw_iid(item_id)} с прогнозируемым рейтингом {rating:.2f}")
else:
    print(f"Пользователь {user_id_to_check} отсутствует в исходных данных")
    print(f"Диапазон ID пользователей: {data['user_id'].min()} - {data['user_id'].max()}")

Пользователь 214 найден в данных
Количество рецензий пользователя 214: 93
Топ-10 фильмов, рекомендованных для пользователя:
Фильм 657 с прогнозируемым рейтингом 4.57
Фильм 474 с прогнозируемым рейтингом 4.56
Фильм 178 с прогнозируемым рейтингом 4.51
Фильм 480 с прогнозируемым рейтингом 4.51
Фильм 175 с прогнозируемым рейтингом 4.49
Фильм 493 с прогнозируемым рейтингом 4.47
Фильм 59 с прогнозируемым рейтингом 4.43
Фильм 488 с прогнозируемым рейтингом 4.42
Фильм 923 с прогнозируемым рейтингом 4.41
Фильм 515 с прогнозируемым рейтингом 4.41


Висновок

У роботі реалізовано рекомендаційну систему на основі датасету MovieLens 100k. Проведено порівняння двох алгоритмів колаборативної фільтрації: SVD та KNNBasic. За допомогою GridSearchCV підібрано оптимальні параметри моделей